# Test quota OpenAI — Embeddings

Vérifie rapidement si ta clé API OpenAI peut appeler l'API embeddings.
Lance les cellules dans l'ordre.

In [ ]:
# Charge la clé depuis le .env (à la racine du projet)
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("../../.env")  # depuis rd/
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY", "")
print(f"Clé trouvée : {'✅ ' + api_key[:8] + '...' if api_key else '❌ OPENAI_API_KEY absente'}")

In [ ]:
# Test 1 — Appel embeddings minimal
from openai import OpenAI

client = OpenAI(api_key=api_key)

try:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input="test",
    )
    vec = response.data[0].embedding
    print(f"✅ Embeddings OK — dimension : {len(vec)}, tokens utilisés : {response.usage.total_tokens}")
except Exception as e:
    print(f"❌ Erreur : {e}")

In [ ]:
# Test 2 — Infos compte (modèles disponibles = indicateur de quota valide)
try:
    models = client.models.list()
    embedding_models = [m.id for m in models.data if "embedding" in m.id]
    print(f"✅ Compte valide — modèles embedding disponibles :")
    for m in embedding_models:
        print(f"   - {m}")
except Exception as e:
    print(f"❌ Erreur : {e}")

In [ ]:
# Test 3 — Embed un texte plus long (simuler une requête RAG réelle)
texte = """
Notre approche méthodologique s'articule autour des cinq critères OCDE/CAD.
La mission inclut des entretiens terrain dans les régions de Souss-Massa et Drâa-Tafilalet.
"""

try:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texte.strip(),
    )
    print(f"✅ Embedding texte long OK — {response.usage.total_tokens} tokens")
    print(f"   Coût estimé : ${response.usage.total_tokens / 1_000_000 * 0.02:.6f}")
except Exception as e:
    print(f"❌ Erreur : {e}")